# Preprocessing - LH/RH Feature Table (Local CPU/GPU)

This notebook filters encoding data, engineers chain features, pivots to wide format, and saves features_lh_rh.csv.
Outputs are saved under results_local_cpu_gpu/preprocessing.
Default run mode is smoke for quick validation.

In [18]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Optional: processing modules as reference hooks
import sys
PROCESSING_ROOT = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/EEG_web")
if str(PROCESSING_ROOT) not in sys.path:
    sys.path.insert(0, str(PROCESSING_ROOT))
try:
    from processing.loader import EEGLoader  # noqa: F401
    from processing.features import EEGFeatures  # noqa: F401
    print("Processing modules available.")
except Exception as e:
    print(f"Processing module import skipped: {e}")

# 1) CONFIGURATION
BASE_DIR = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook")
ENCODING_CSV = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/dataset/encoding chaining.csv")
OUTPUT_DIR = BASE_DIR / "results_local_cpu_gpu" / "preprocessing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "features_lh_rh.csv"
COMPAT_OUTPUT_CSV = BASE_DIR / "features_lh_rh.csv"

TARGET_SCENARIOS = [1, 2]
TARGET_TASKS = ["Thinking", "Acting"]
MOTOR_CHANNELS = ["C3", "Cz", "C4", "FC3", "FC4", "CP3", "CP4"]
TARGET_SUBBANDS = ["Alpha", "Beta", "Gamma"]

RUN_MODE = "smoke"  # change to "full" for full execution
SMOKE_MAX_ROWS = 120000

def savefig(name: str):
    plt.tight_layout()
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

# 2) LOAD CSV + INVENTORY
raw_df = pd.read_csv(ENCODING_CSV)
if RUN_MODE == "smoke" and len(raw_df) > SMOKE_MAX_ROWS:
    raw_df = raw_df.sample(SMOKE_MAX_ROWS, random_state=42).reset_index(drop=True)

if "scenario_id" in raw_df.columns:
    raw_df["scenario_id"] = pd.to_numeric(raw_df["scenario_id"], errors="coerce")

print(f"Raw shape: {raw_df.shape}")
for col in raw_df.columns:
    nunq = raw_df[col].nunique(dropna=True)
    print(f"{col}: unique={nunq}")

# 3) FILTER STEP-BY-STEP
filtered = raw_df.copy()
print("\nFilter progression:")
print("Start:", filtered.shape)

mask_healthy = filtered["subject_id"].astype(str).str.startswith("id")
filtered = filtered[mask_healthy].copy()
print("Healthy only:", filtered.shape)

filtered = filtered[filtered["scenario_id"].isin(TARGET_SCENARIOS)].copy()
print("Scenario 1/2:", filtered.shape)

filtered = filtered[filtered["task"].isin(TARGET_TASKS)].copy()
print("Thinking/Acting:", filtered.shape)

filtered = filtered[filtered["channel"].isin(MOTOR_CHANNELS)].copy()
print("Motor channels:", filtered.shape)

filtered = filtered[filtered["subband"].isin(TARGET_SUBBANDS)].copy()
print("Target subbands:", filtered.shape)

# 4) FEATURE ENGINEERING from chain_sequence
def chain_features(chain_seq: str) -> dict:
    bits = [c for c in str(chain_seq) if c in "01"]
    n = len(bits)
    if n == 0:
        return {
            "chain_len": 0,
            "chain_ones": 0,
            "chain_zeros": 0,
            "chain_ones_ratio": np.nan,
            "chain_longest_run1": 0,
            "chain_longest_run0": 0,
            "chain_transitions": 0,
            "chain_entropy": np.nan,
        }

    ones = bits.count("1")
    zeros = n - ones

    transitions = 0
    longest_run1 = 1 if ones > 0 else 0
    longest_run0 = 1 if zeros > 0 else 0
    cur_char = bits[0]
    cur_len = 1

    for i in range(1, n):
        if bits[i] != bits[i - 1]:
            transitions += 1
        if bits[i] == cur_char:
            cur_len += 1
        else:
            if cur_char == "1":
                longest_run1 = max(longest_run1, cur_len)
            else:
                longest_run0 = max(longest_run0, cur_len)
            cur_char = bits[i]
            cur_len = 1

    if cur_char == "1":
        longest_run1 = max(longest_run1, cur_len)
    else:
        longest_run0 = max(longest_run0, cur_len)

    p1 = ones / n
    p0 = zeros / n
    entropy = 0.0
    for p in [p0, p1]:
        if p > 0:
            entropy += -(p * np.log2(p))

    return {
        "chain_len": n,
        "chain_ones": ones,
        "chain_zeros": zeros,
        "chain_ones_ratio": p1,
        "chain_longest_run1": longest_run1,
        "chain_longest_run0": longest_run0,
        "chain_transitions": transitions,
        "chain_entropy": entropy,
    }

chain_feat_df = filtered["chain_sequence"].apply(chain_features).apply(pd.Series)
filtered = pd.concat([filtered.reset_index(drop=True), chain_feat_df.reset_index(drop=True)], axis=1)

if "chain_ratio" in filtered.columns:
    filtered["chain_ratio"] = pd.to_numeric(filtered["chain_ratio"], errors="coerce")

# 5) BUILD LABELS
filtered["label"] = np.where(filtered["scenario_id"] == 1, 0, 1)
filtered["label_name"] = filtered["label"].map({0: "LH", 1: "RH"})

# 6) IDENTIFY NUMERIC FEATURES
META_COLS = {
    "subject_id", "scenario", "scenario_id", "filename", "task", "channel",
    "subband", "feature", "chain_sequence", "label", "label_name"
}
numeric_cols = filtered.select_dtypes(include=[np.number]).columns.tolist()
NUMERIC_FEATS = [c for c in numeric_cols if c not in META_COLS]
print(f"Numeric features for pivot: {len(NUMERIC_FEATS)}")
print(NUMERIC_FEATS[:20])

# 7) PIVOT TO WIDE FORMAT
GROUP_KEY = ["subject_id", "scenario_id", "filename", "task", "label", "label_name"]
pivot_parts = []
for feat in NUMERIC_FEATS:
    pv = filtered.pivot_table(
        index=GROUP_KEY,
        columns=["channel", "subband"],
        values=feat,
        aggfunc="mean"
    )
    if pv.empty:
        continue
    pv.columns = [f"{ch}_{sb}_{feat}" for ch, sb in pv.columns]
    pivot_parts.append(pv)

if not pivot_parts:
    raise ValueError("No pivot parts generated. Check filtering constraints and input data.")

wide_df = pd.concat(pivot_parts, axis=1).reset_index()
wide_df = wide_df.loc[:, ~wide_df.columns.duplicated()].copy()

# 8) MISSING VALUES
feature_cols = [c for c in wide_df.columns if c not in GROUP_KEY]
missing_pct = wide_df[feature_cols].isna().mean() * 100
to_drop = missing_pct[missing_pct > 50.0].index.tolist()
if to_drop:
    wide_df = wide_df.drop(columns=to_drop)

feature_cols = [c for c in wide_df.columns if c not in GROUP_KEY]
wide_df[feature_cols] = wide_df[feature_cols].apply(pd.to_numeric, errors="coerce")
wide_df[feature_cols] = wide_df[feature_cols].fillna(wide_df[feature_cols].median(numeric_only=True))

miss_matrix = wide_df[feature_cols].isna().iloc[:250, :60] if feature_cols else pd.DataFrame()
if not miss_matrix.empty:
    plt.figure(figsize=(14, 5))
    sns.heatmap(miss_matrix, cbar=False)
    plt.title("Missing Value Heatmap (first 250 rows x first 60 features)")
    savefig("missing_values_heatmap_first60.png")

# 9) CLASS BALANCE CHECK
class_counts = wide_df["label_name"].value_counts().reindex(["LH", "RH"]).fillna(0)
plt.figure(figsize=(7, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="Set2")
plt.title("Class Balance (Wide Dataset)")
for i, v in enumerate(class_counts.values):
    plt.text(i, v, str(int(v)), ha="center", va="bottom")
savefig("class_balance_overall.png")

task_counts = wide_df.groupby(["task", "label_name"]).size().reset_index(name="count")
plt.figure(figsize=(9, 4))
sns.barplot(data=task_counts, x="task", y="count", hue="label_name", palette="Set1")
plt.title("Class Balance by Task")
savefig("class_balance_by_task.png")

# 10) FEATURE STATISTICS
desc = wide_df[feature_cols].describe().T if feature_cols else pd.DataFrame()
if not desc.empty:
    desc["cv"] = desc["std"] / desc["mean"].replace(0, np.nan)
    desc = desc.sort_values("std", ascending=False)
    desc.head(10).to_csv(OUTPUT_DIR / "top10_features_by_std.csv")

# 11) CORRELATION WITH LABEL
corr_rows = []
for col in feature_cols:
    vals = pd.to_numeric(wide_df[col], errors="coerce")
    ok = vals.notna()
    if ok.sum() < 6:
        continue
    r, p = pointbiserialr(wide_df.loc[ok, "label"], vals.loc[ok])
    if np.isfinite(r):
        corr_rows.append({"feature": col, "correlation": r, "p_value": p, "abs_correlation": abs(r)})

corr_df = pd.DataFrame(corr_rows).sort_values("abs_correlation", ascending=False) if corr_rows else pd.DataFrame(columns=["feature", "correlation", "p_value", "abs_correlation"])
corr_df.to_csv(OUTPUT_DIR / "feature_label_correlation.csv", index=False)

top20 = corr_df.head(20).sort_values("abs_correlation", ascending=True)
if not top20.empty:
    plt.figure(figsize=(10, 8))
    colors = ["tab:blue" if v >= 0 else "tab:red" for v in top20["correlation"]]
    plt.barh(top20["feature"], top20["correlation"], color=colors)
    plt.title("Top 20 Features by |Point-Biserial Correlation|")
    plt.xlabel("Correlation with label (RH=1)")
    savefig("top20_feature_correlation.png")

# 12) SAVE OUTPUTS
wide_df.to_csv(OUTPUT_CSV, index=False)
wide_df.to_csv(COMPAT_OUTPUT_CSV, index=False)

summary = pd.DataFrame([
    {"metric": "run_mode", "value": RUN_MODE},
    {"metric": "raw_rows", "value": len(raw_df)},
    {"metric": "filtered_rows", "value": len(filtered)},
    {"metric": "wide_rows", "value": len(wide_df)},
    {"metric": "wide_columns", "value": wide_df.shape[1]},
    {"metric": "n_subjects", "value": wide_df["subject_id"].nunique()},
    {"metric": "class_lh", "value": int((wide_df["label"] == 0).sum())},
    {"metric": "class_rh", "value": int((wide_df["label"] == 1).sum())},
    {"metric": "n_numeric_features", "value": len(feature_cols)},
])
summary.to_csv(OUTPUT_DIR / "preprocessing_summary.csv", index=False)

print(f"Final wide shape: {wide_df.shape}")
print("Class counts:")
print(wide_df["label_name"].value_counts())
print("Saved files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")
print(f"Compatibility copy: {COMPAT_OUTPUT_CSV}")

Processing modules available.
Raw shape: (120000, 10)
subject_id: unique=150
scenario: unique=9
scenario_id: unique=9
filename: unique=1350
task: unique=4
channel: unique=3
subband: unique=4
feature: unique=6
chain_sequence: unique=99492
chain_ratio: unique=2997

Filter progression:
Start: (120000, 10)
Healthy only: (120000, 10)
Scenario 1/2: (28203, 10)
Thinking/Acting: (7096, 10)
Motor channels: (7096, 10)
Target subbands: (1723, 10)
Numeric features for pivot: 9
['chain_ratio', 'chain_len', 'chain_ones', 'chain_zeros', 'chain_ones_ratio', 'chain_longest_run1', 'chain_longest_run0', 'chain_transitions', 'chain_entropy']
Final wide shape: (300, 33)
Class counts:
label_name
LH    150
RH    150
Name: count, dtype: int64
Saved files:
- class_balance_by_task.png (36.2 KB)
- class_balance_overall.png (31.9 KB)
- feature_label_correlation.csv (2.3 KB)
- features_lh_rh.csv (96.0 KB)
- missing_values_heatmap_first60.png (199.9 KB)
- preprocessing_summary.csv (0.2 KB)
- top10_features_by_std.c